In [16]:
import sys
sys.path.append("../")

In [17]:
import torch
from qmpsqsc.models import mpsqsc
from qmpsqsc.models import qmps
from importlib import reload

reload(mpsqsc)

<module 'qmpsqsc.models.mpsqsc' from '/Users/keisuke/Documents/projects/mps4qsc/notebooks/../qmpsqsc/models/mpsqsc/__init__.py'>

In [23]:
import torch.nn.functional as F

L = 100
chi = 2
d = 2
qscr = mpsqsc.MpsQsc(L, chi, d)
allup = torch.zeros(L, d, dtype = torch.complex128)
allup[:, 0] = 1.0
alldown = torch.zeros(L, d, dtype = torch.complex128)
alldown[:, 1] = 1.0

mps_allup = mpsqsc.build_product_state(L, d, allup)
mps_alldown = mpsqsc.build_product_state(L, d, alldown)

mps_allup = mps_allup.normalize()
mps_alldown = mps_alldown.normalize()

mpsghz = mpsqsc.build_ghz_state(L, d, chi, dtype = torch.complex128)
mpsghz = mpsghz.normalize()
qsc = mpsqsc.build_qsc_from_mpstate(mpsghz)

logits_ghz = qsc.contract_with_state(mpsghz)
logits_allup = qsc.contract_with_state(mps_allup)
logits_alldown = qsc.contract_with_state(mps_alldown)

logits_ghz, logits_allup, logits_alldown

(tensor([1.0000+0.j, 0.0000+0.j], dtype=torch.complex128,
        grad_fn=<ViewBackward0>),
 tensor([0.7071+0.j, 0.0000+0.j], dtype=torch.complex128,
        grad_fn=<ViewBackward0>),
 tensor([0.7071+0.j, 0.0000+0.j], dtype=torch.complex128,
        grad_fn=<ViewBackward0>))

In [24]:
qscr = qscr.canonicalize(truncate=True)

us, last = qmps.construct_unitary_from_As(qscr.As)
u0 = torch.kron(us[0].contiguous(), torch.eye(d, dtype = us[0].dtype))
us[1] = us[1] @ u0



qmpsqsc = qmps.qMPS(L, chi, d, Us=us[1:], last_unitary=last)

In [25]:
# qmpsqsc._build_equation_circuit()

In [28]:
qmpsqsc._contract_circuit_with_state(mpsghz)

tensor([[ 0.4209-3.2517e-17j, -0.2049+4.8998e-02j],
        [-0.2049-4.8998e-02j,  0.5791-6.0511e-17j]], dtype=torch.complex128,
       grad_fn=<ViewBackward0>)

In [22]:
qmpsqsc.last_unitary @ qmpsqsc.last_unitary.mH

tensor([[[[ 7.3114e-01+0.0000e+00j,  2.5875e-01-5.7327e-03j],
          [ 2.5875e-01+5.7327e-03j,  2.6886e-01+2.6251e-18j]],

         [[ 2.8391e-01+9.2779e-19j, -1.5707e-01-1.5268e-02j],
          [-1.5707e-01+1.5268e-02j,  7.1609e-01-2.6251e-18j]]],


        [[[ 1.1102e-16-3.4191e-51j, -8.9842e-09+3.4051e-18j],
          [-8.9842e-09-3.4051e-18j,  1.0000e+00-1.0417e-18j]],

         [[ 9.8495e-01+3.4191e-51j, -1.0167e-01+2.1001e-02j],
          [-1.0167e-01-2.1001e-02j,  1.5052e-02+1.1390e-19j]]]],
       dtype=torch.complex128, grad_fn=<UnsafeViewBackward0>)

In [9]:
mpsghz = mpsqsc.build_ghz_state(L, d, chi)
mpsghz2 = mpsqsc.build_ghz_state(L, d, chi)
As = mpsghz2.As
As[0][:, 1] = -As[0][:, 1]
mpsghz2.set_As(As)
qsc = mpsqsc.build_2qsc_from_mpstate(mpsghz, mpsghz2)

In [5]:
ghz3 = mpsqsc.build_ghz_state(L, d, chi)
ghz3.As[0][:] = torch.tensor([[0, 1], [1, 0]])

In [6]:
qsc_can = qsc.canonicalize(truncate=True)

In [8]:
qsc_can.As[2].shape

torch.Size([4, 2, 4])

In [30]:
qsc2.As[2].shape

torch.Size([2, 2, 2])

In [11]:
qsc2.contract_with_state(mpsghz)

tensor([2.0000e+00, 2.3912e-30], dtype=torch.float64, grad_fn=<ViewBackward0>)

In [12]:
import random
from typing import List, Tuple, Dict
import torch

@torch.no_grad()
def _ensure_same_device_dtype(state, device, dtype):
    if getattr(state, "device", None) != device or getattr(state, "dtype", None) != dtype:
        state.to(device=device, dtype=dtype)
    return state

def _amps_to_probs_batch(amps: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    """
    Convert a batch of amplitudes to Born probabilities.
      amps: (B, 2) real or complex
      returns probs: (B, 2), each row sums to 1
    """
    if torch.is_complex(amps):
        abs_sq = (amps.conj() * amps).real
    else:
        abs_sq = amps * amps
    denom = abs_sq.sum(dim=-1, keepdim=True).clamp_min(eps)
    return abs_sq / denom

def train_classifier_qsc_on_ghz_vs_rho(
    qsc,
    mpsghz,
    mps_allup,
    mps_alldown,
    *,
    steps: int = 2000,
    lr: float = 1e-2,
    batch_size: int = 16,          # kept for API compatibility but ignored
    ghz_fraction: float = 0.5,     # ignored (fixed composition batch)
    weight_decay: float = 0.0,
    grad_clip: float | None = None,
    seed: int | None = 0,
    log_every: int = 100,
) -> Dict[str, List[float]]:
    """
    Train qsc so that it outputs class 1 for GHZ, class 0 for the mixture ρ.

    CHANGE: No softmax. We treat the 2-D output as amplitudes a,
            compute probabilities p_k = |a_k|^2 / sum_j |a_j|^2,
            and minimize NLL:  -log p_y.

    Each step uses a fixed batch of 4 samples:
        [GHZ, GHZ, all-up, all-down] with labels [1, 1, 0, 0].
    """
    if seed is not None:
        torch.manual_seed(seed)
        random.seed(seed)

    device, dtype = qsc.device, qsc.dtype

    # Ensure states match qsc's device/dtype
    _ensure_same_device_dtype(mpsghz, device, dtype)
    _ensure_same_device_dtype(mps_allup, device, dtype)
    _ensure_same_device_dtype(mps_alldown, device, dtype)

    # Collect parameters (works for nn.Module or your minimal class exposing .parameters())
    try:
        params = list(qsc.parameters())
        if len(params) == 0:
            params = list(getattr(qsc, "As", []))
    except Exception:
        params = list(getattr(qsc, "As", []))

    optim = torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)

    history: Dict[str, List[float]] = {"loss": [], "acc": []}

    # Fixed batch content per step
    fixed_states = [mpsghz, mpsghz, mps_allup, mps_alldown]
    fixed_labels = torch.tensor([0, 0, 1, 1], dtype=torch.long, device=device)

    for step in range(1, steps + 1):
        if hasattr(qsc, "train"):
            qsc.train()

        optim.zero_grad(set_to_none=True)

        # Optionally shuffle the 4 examples each step
        idx = [0, 1, 2, 3]
        random.shuffle(idx)
        states = [fixed_states[i] for i in idx]
        y = fixed_labels[idx]                       # shape (4,)

        # ---- forward: amplitudes -> Born probs ----
        amps_list = [qsc.contract_with_state(st) for st in states]   # each (2,)
        amps_batch = torch.stack(amps_list, dim=0)                   # (4, 2)
        probs = _amps_to_probs_batch(amps_batch)                     # (4, 2)
        print(probs)

        # ---- loss: negative log-likelihood on target class ----
        eps = 1e-12
        p_y = probs.gather(1, y.view(-1, 1)).squeeze(1)              # (4,)
        nll = -torch.log(p_y.clamp_min(eps)).mean()
        loss = nll

        # ---- backward/step ----
        loss.backward()
        if grad_clip is not None and len(params) > 0:
            torch.nn.utils.clip_grad_norm_(params, max_norm=grad_clip)
        optim.step()

        # ---- metric: accuracy by argmax over Born probs ----
        with torch.no_grad():
            pred = probs.argmax(dim=-1)                              # (4,)
            acc = (pred == y).float().mean().item()

        history["loss"].append(float(loss.item()))
        history["acc"].append(acc)

        if (log_every is not None) and (step % log_every == 0):
            print(f"[step {step:5d}] loss={loss.item():.6f}  acc={acc:.3f}")

    return history


In [18]:
# You already created these:
# L = 10; chi = 2; d = 2
# qsc = mps_classifier.MpsQsc(L, chi, d)
# mps_allup, mps_alldown, mpsghz prepared and normalized
# qsc = mps_classifier.build_qsc_from_mpstate(mpsghz)   # if you prefer that init

hist = train_classifier_qsc_on_ghz_vs_rho(
    qsc_2,
    mpsghz=mpsghz,
    mps_allup=mps_allup,
    mps_alldown=mps_alldown,
    steps=150,
    lr=1e-3,
    weight_decay=0.0,
    grad_clip=1.0,   # optional
    seed=0,
    log_every=1,
)

tensor([[3.0346e-06, 1.0000e+00],
        [1.0000e+00, 7.1195e-25],
        [1.0000e+00, 7.1195e-25],
        [3.0346e-06, 1.0000e+00]], dtype=torch.float64, grad_fn=<DivBackward0>)
[step     1] loss=0.000002  acc=1.000
tensor([[3.5493e-01, 6.4507e-01],
        [3.0151e-06, 1.0000e+00],
        [3.0151e-06, 1.0000e+00],
        [3.5493e-01, 6.4507e-01]], dtype=torch.float64, grad_fn=<DivBackward0>)
[step     2] loss=0.517925  acc=0.500
tensor([[3.0295e-06, 1.0000e+00],
        [5.9704e-04, 9.9940e-01],
        [3.0295e-06, 1.0000e+00],
        [5.9704e-04, 9.9940e-01]], dtype=torch.float64, grad_fn=<DivBackward0>)
[step     3] loss=3.711762  acc=0.500
tensor([[3.0424e-06, 1.0000e+00],
        [6.8192e-04, 9.9932e-01],
        [3.0423e-06, 1.0000e+00],
        [6.8192e-04, 9.9932e-01]], dtype=torch.float64, grad_fn=<DivBackward0>)
[step     4] loss=3.645298  acc=0.500
tensor([[3.0580e-06, 1.0000e+00],
        [3.2454e-03, 9.9675e-01],
        [3.0579e-06, 1.0000e+00],
        [3.2454e-0

KeyboardInterrupt: 

In [14]:
qsc_2 = qsc.truncate_bond_dimension(2)

In [19]:
qsc_2.save_to("data/qsc_2.pth")

In [21]:
blob = torch.load("data/qsc_2.pth")

In [22]:
blob

{'format': 'mps_v1',
 'class_name': 'MpsQsc',
 'L': 100,
 'd': 2,
 'chi': 2,
 'out_dim': 2,
 'optimize': 'random-greedy',
 'requires_grad': True,
 'training': True,
 'As': [tensor([[0.9998, 0.0000],
          [0.0000, 1.0002]], dtype=torch.float64),
  tensor([[[0.0000e+00, 9.9977e-01],
           [0.0000e+00, 0.0000e+00]],
  
          [[2.2204e-16, 0.0000e+00],
           [1.0002e+00, 0.0000e+00]]], dtype=torch.float64),
  tensor([[[ 0.0000,  0.0000],
           [ 0.0000,  1.0002]],
  
          [[-0.9998,  0.0000],
           [ 0.0000,  0.0000]]], dtype=torch.float64),
  tensor([[[ 0.0000e+00,  9.9977e-01],
           [ 0.0000e+00,  0.0000e+00]],
  
          [[ 2.2204e-16,  0.0000e+00],
           [-1.0002e+00,  0.0000e+00]]], dtype=torch.float64),
  tensor([[[0.0000, 0.0000],
           [0.0000, 1.0002]],
  
          [[0.9998, 0.0000],
           [0.0000, 0.0000]]], dtype=torch.float64),
  tensor([[[0.9998, 0.0000],
           [0.0000, 0.0000]],
  
          [[0.0000, 0.0000],
   